# 03 · Simple RAG: answer from your own documents

A model only knows what it learned in training. It does **not** know your shop's refund policy, and if you ask, it **guesses**.

**RAG** (Retrieval-Augmented Generation) fixes this in three steps:

```
Question ──> 1. RETRIEVE the most relevant chunks of your documents
         ──> 2. AUGMENT the prompt: "Answer using only this text: ..."
         ──> 3. GENERATE the answer
```

The model reads your text *at the moment it answers*, so there is no retraining. If the policy changes, you just update the file.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("Ready")

Ready


## Step 0: prepare the knowledge base
The folder `knowledge_base/` holds four small policy files. We split each file into paragraphs. Each paragraph is a **chunk**, one idea that can be found on its own.

Why chunks and not whole files? Smaller pieces find more precise matches, and we only send the relevant few to the model.

In [2]:
import re
from pathlib import Path

def load_chunks():
    """Read every .txt file in knowledge_base/ and split it into paragraphs. Each paragraph is a 'chunk'."""
    chunks = []
    for file in sorted(Path("knowledge_base").glob("*.txt")):
        text = file.read_text(encoding="utf-8")
        for paragraph in re.split(r"\n\s*\n", text):    # a blank line separates paragraphs
            if paragraph.strip():
                chunks.append({"source": file.name, "text": paragraph.strip()})
    return chunks

chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks from the knowledge base")

Loaded 15 chunks from the knowledge base


In [3]:
print("Example chunk:")
print(chunks[0])

Example chunk:
{'source': 'orders_and_payments.txt', 'text': 'Payment methods: We accept credit cards, debit cards and UPI.'}


## Step 1: retrieve
We reuse the `embed` and `similarity` functions from the last notebook. Every chunk is embedded **once**. For each question we embed only the question and pick the closest chunks.

In [4]:
def embed(texts):
    """Turn a list of texts into a list of vectors (a vector is just a long list of numbers)."""
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in response.data]

def similarity(vector_a, vector_b):
    """How close in meaning are two texts? Higher = closer.
    OpenAI vectors all have length 1, so multiplying them pair by pair and adding up is enough."""
    return sum(a * b for a, b in zip(vector_a, vector_b))

In [5]:
# Embed every chunk ONCE and keep the vectors. We reuse them for every question.
chunk_vectors = embed([chunk["text"] for chunk in chunks])

def retrieve(question, top_k=3):
    """Find the top_k chunks whose meaning is closest to the question."""
    question_vector = embed([question])[0]
    scored = [(similarity(question_vector, vector), chunk) for vector, chunk in zip(chunk_vectors, chunks)]
    scored.sort(key=lambda pair: pair[0], reverse=True)    # best match first
    return scored[:top_k]

In [6]:
for score, chunk in retrieve("What is the fee for returning an opened item?"):
    print(f"{score:.2f}  [{chunk['source']}]  {chunk['text'][:90]}...")

0.66  [returns_policy.txt]  Opened items: A faulty item can be returned even if it is opened. If an opened item is not...
0.51  [returns_policy.txt]  Return shipping: Return shipping is free for faulty items. For any other return, the custo...
0.40  [returns_policy.txt]  How refunds are paid: Refunds go back to the original payment method. The money usually ar...


## Without RAG: the model guesses
First a question whose answer is specific to **our** shop.

In [7]:
question = "What is the restocking fee if I return an opened item that is not faulty?"

response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}])
print(response.choices[0].message.content)

Restocking fees for opened items can vary depending on the retailer's policy. Generally, restocking fees can range from 10% to 30% of the item's price. Some retailers might not charge a restocking fee at all, while others might charge more for certain types of items, such as electronics. It's best to check the specific return policy of the retailer from which you purchased the item for accurate details on their restocking fees.


The model gives a general answer such as "it depends on the store". It has no idea what **our** fee is.

## With RAG: retrieve, augment, generate
The prompt now contains the retrieved chunks, and an instruction to **use only them** and to say so if the answer is not there.

In [8]:
def answer_with_rag(question):
    # 1. RETRIEVE the 3 most relevant chunks
    results = retrieve(question, top_k=3)
    context = "\n".join(f"[{chunk['source']}] {chunk['text']}" for score, chunk in results)

    # 2. AUGMENT the prompt with those chunks
    prompt = f"""Answer the customer's question using ONLY the information below.
If the answer is not there, say you don't know and offer to pass them to a human agent.

Information:
{context}

Question: {question}"""

    # 3. GENERATE the answer
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}])
    answer = response.choices[0].message.content
    sources = sorted({chunk["source"] for score, chunk in results})
    return answer, sources

answer, sources = answer_with_rag("What is the restocking fee if I return an opened item that is not faulty?")
print(answer)
print("\nSources:", sources)

The restocking fee for returning an opened item that is not faulty is 10%.

Sources: ['returns_policy.txt']


Now the answer is **10%**, taken from our policy, and we can show the customer **where it came from**.

## A question that is not in the documents
A good RAG system does not make things up. Ask something our documents do not cover:

In [9]:
answer, sources = answer_with_rag("Do you match prices from other shops?")
print(answer)

I don't know. Would you like me to pass you to a human agent?


The model says it does not know. That is exactly what you want from a support assistant.

## Try a few more

In [10]:
for question in [
    "How long does express shipping take and what does it cost?",
    "My laptop is 8 months old and the screen has a fault. Is it covered?",
]:
    answer, sources = answer_with_rag(question)
    print(f"Q: {question}\nA: {answer}\n   Sources: {sources}\n")

Q: How long does express shipping take and what does it cost?
A: Express shipping takes 1 to 2 business days and always costs $12.
   Sources: ['shipping_policy.txt', 'warranty.txt']



Q: My laptop is 8 months old and the screen has a fault. Is it covered?
A: Yes, your laptop is covered by the warranty since it is within the 12-month warranty period for electronics.
   Sources: ['returns_policy.txt', 'warranty.txt']



## Recap
* RAG = **retrieve** relevant chunks, **augment** the prompt with them, **generate** the answer.
* It gives the model knowledge it was never trained on, and you update it by editing a file.
* Telling the model to "use only this information" reduces made-up answers.
* Returning the **sources** lets people check the answer. (Our list shows every file the retrieved chunks came from, so it can include a file the answer did not actually need.)

> ⚠️ Retrieval quality decides answer quality. If the wrong chunks are found, the answer will be wrong, however good the model is. Always test with real questions.

Next: **long-term memory**, so the assistant remembers a customer between conversations.